# 基于传统ML的图像像素识别

Image Loading

In [28]:
import cv2
import numpy as np
from PIL import Image
import os
import torch
import matplotlib.pyplot as plt
from os.path import join


In [9]:
# Function to binarize a tensor of images using Otsu's method
def binarize_tensor_images(tensor_images):
    binary_images = []
    for image in tensor_images:
        # Convert PyTorch tensor to numpy array
        image_np = image.numpy().squeeze()  # Remove single-dimensional entries
        
        # Ensure the image is in uint8 format
        if image_np.dtype != np.uint8:
            image_np = (image_np * 255).astype(np.uint8)
        
        # Apply Otsu's thresholding
        _, binary_image = cv2.threshold(image_np, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        binary_images.append(torch.tensor(binary_image))
    return torch.stack(binary_images)

In [10]:
# Function to load and binarize images in batches
def process_images_in_batches(data_path, batch_size=100):
    img_list = []

    for file in os.listdir(data_path):
        if not file.endswith('seg_data.pt'):
            continue
        img_path = join(data_path, file)
        data = torch.load(img_path)
        img_list.append(data)
        print(f"Loaded {file}")

    img_list = torch.cat(img_list)
    print("Loaded images shape:", img_list.shape)
    print("Data type of loaded images:", img_list.dtype)

    binarized_list = []

    # Process images in batches
    num_batches = len(img_list) // batch_size + 1
    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, len(img_list))
        batch = img_list[start_idx:end_idx]
        binarized_batch = binarize_tensor_images(batch)
        binarized_list.append(binarized_batch)
        print(f"Processed batch {i+1}/{num_batches}")

    binarized_images = torch.cat(binarized_list)
    return binarized_images

In [11]:
CVL_path ='/root/autodl-tmp/APS360_Project/Datasets/CVL_Processed'
IAM_path = '/root/autodl-tmp/APS360_Project/Datasets/IAM_Processed'

In [20]:
cvlt_binarized_images = process_images_in_batches(CVL_path, batch_size=100)
torch.save(cvlt_binarized_images, '/root/autodl-tmp/APS360_Project/Machine_Learning_Output/binary_image_bounding_box/CVL_binary.pt')
print("Saved binarized CVL images")

Loaded seg_data.pt
Loaded images shape: torch.Size([1598, 1, 1024, 1024])
Data type of loaded images: torch.float32
Processed batch 1/16
Processed batch 2/16
Processed batch 3/16
Processed batch 4/16
Processed batch 5/16
Processed batch 6/16
Processed batch 7/16
Processed batch 8/16
Processed batch 9/16
Processed batch 10/16
Processed batch 11/16
Processed batch 12/16
Processed batch 13/16
Processed batch 14/16
Processed batch 15/16
Processed batch 16/16
Saved binarized CVL images


In [13]:
# Function to invert and binarize a tensor of images using Otsu's method
def binarize_tensor_images_IAM(tensor_images):
    binary_images = []
    for image in tensor_images:
        # Convert PyTorch tensor to numpy array
        image_np = image.numpy().squeeze()  # Remove single-dimensional entries
        
        # Ensure the image is in uint8 format
        if image_np.dtype != np.uint8:
            image_np = (image_np * 255).astype(np.uint8)
        
        # Invert the image
        image_np = 255 - image_np
        
        # Apply Otsu's thresholding
        _, binary_image = cv2.threshold(image_np, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        binary_images.append(torch.tensor(binary_image, dtype=torch.uint8))
    return torch.stack(binary_images)

# Function to process images in batches
def process_images_in_batches_IAM(data_path, batch_size=100):
    img_list = []

    for file in os.listdir(data_path):
        if not file.endswith('seg_data.pt'):
            continue
        img_path = join(data_path, file)
        data = torch.load(img_path)
        img_list.append(data)
        print(f"Loaded {file}")

    img_list = torch.cat(img_list)
    print("Loaded images shape:", img_list.shape)
    print("Data type of loaded images:", img_list.dtype)

    binarized_list = []

    # Process images in batches
    num_batches = len(img_list) // batch_size + 1
    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, len(img_list))
        batch = img_list[start_idx:end_idx]
        binarized_batch = binarize_tensor_images_IAM(batch)
        binarized_list.append(binarized_batch)
        print(f"Processed batch {i+1}/{num_batches}")

    binarized_images = torch.cat(binarized_list)
    return binarized_images

In [14]:
# Process IAM images
iam_binarized_images = process_images_in_batches_IAM(IAM_path, batch_size=100)
torch.save(iam_binarized_images, '/root/autodl-tmp/APS360_Project/Machine_Learning_Output/binary_image_bounding_box/IAM_binary.pt')
print("Saved binarized IAM images")

Loaded seg_data.pt
Loaded images shape: torch.Size([1539, 1, 1024, 1024])
Data type of loaded images: torch.float32
Processed batch 1/16
Processed batch 2/16
Processed batch 3/16
Processed batch 4/16
Processed batch 5/16
Processed batch 6/16
Processed batch 7/16
Processed batch 8/16
Processed batch 9/16
Processed batch 10/16
Processed batch 11/16
Processed batch 12/16
Processed batch 13/16
Processed batch 14/16
Processed batch 15/16
Processed batch 16/16
Saved binarized IAM images


Connected Component Detection

In [21]:
def connected_component_labeling_batch(tensor_images):
    results = []
    for image in tensor_images:
        image_np = image.numpy()
        # Apply connected component labeling
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(image_np, connectivity=8)
        results.append((num_labels, labels, stats, centroids))
    return results

def process_in_batches(tensor_images, batch_size, processing_fn, output_path):
    processed_batches = []
    num_batches = len(tensor_images) // batch_size + (1 if len(tensor_images) % batch_size != 0 else 0)
    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, len(tensor_images))
        batch = tensor_images[start_idx:end_idx]
        processed_batch = processing_fn(batch)
        processed_batches.extend(processed_batch)
        
        # Save intermediate results to avoid memory overflow
        torch.save(processed_batches, output_path)
        processed_batches = []  # Clear list to free memory
        
        print(f"Processed batch {i + 1}/{num_batches}")
        
        # Free GPU memory if necessary
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return processed_batches

In [4]:
output_dir = '/root/autodl-tmp/APS360_Project/Machine_Learning_Output/binary_image_bounding_box'

# Load binarized images
CVL_binarized_images_path = '/root/autodl-tmp/APS360_Project/Machine_Learning_Output/binary_image_bounding_box/CVL_binary.pt'
IAM_binarized_images_path = '/root/autodl-tmp/APS360_Project/Machine_Learning_Output/binary_image_bounding_box/IAM_binary.pt'

cvl_binarized_images = torch.load(CVL_binarized_images_path)
iam_binarized_images = torch.load(IAM_binarized_images_path)

# Process in smaller batches
batch_size = 50

In [69]:
# CVL dataset
cvl_output_path = os.path.join(output_dir, 'CVL_seg_connected_components.pt')
cvl_labels_stats_centroids = process_in_batches(cvl_binarized_images, batch_size, connected_component_labeling_batch, cvl_output_path)

Processed batch 1/32
Processed batch 2/32
Processed batch 3/32
Processed batch 4/32
Processed batch 5/32
Processed batch 6/32
Processed batch 7/32
Processed batch 8/32
Processed batch 9/32
Processed batch 10/32
Processed batch 11/32
Processed batch 12/32
Processed batch 13/32
Processed batch 14/32
Processed batch 15/32
Processed batch 16/32
Processed batch 17/32
Processed batch 18/32
Processed batch 19/32
Processed batch 20/32
Processed batch 21/32
Processed batch 22/32
Processed batch 23/32
Processed batch 24/32
Processed batch 25/32
Processed batch 26/32
Processed batch 27/32
Processed batch 28/32
Processed batch 29/32
Processed batch 30/32
Processed batch 31/32
Processed batch 32/32


In [70]:
# IAM dataset
iam_output_path = os.path.join(output_dir, 'IAM_seg_connected_components.pt')
iam_labels_stats_centroids = process_in_batches(iam_binarized_images, batch_size, connected_component_labeling_batch, iam_output_path)

print("Connected component detection completed for CVL and IAM datasets.")

Processed batch 1/31
Processed batch 2/31
Processed batch 3/31
Processed batch 4/31
Processed batch 5/31
Processed batch 6/31
Processed batch 7/31
Processed batch 8/31
Processed batch 9/31
Processed batch 10/31
Processed batch 11/31
Processed batch 12/31
Processed batch 13/31
Processed batch 14/31
Processed batch 15/31
Processed batch 16/31
Processed batch 17/31
Processed batch 18/31
Processed batch 19/31
Processed batch 20/31
Processed batch 21/31
Processed batch 22/31
Processed batch 23/31
Processed batch 24/31
Processed batch 25/31
Processed batch 26/31
Processed batch 27/31
Processed batch 28/31
Processed batch 29/31
Processed batch 30/31
Processed batch 31/31
Connected component detection completed for CVL and IAM datasets.


Bounding Box Calculation

In [29]:
# Function to apply connected component labeling and calculate bounding boxes
def calculate_bounding_boxes(tensor_images):
    results = []
    for image_tensor in tensor_images:
        # Convert tensor to numpy array
        image_np = image_tensor.numpy()
        
        # Convert to binary image (assuming already binarized)
        binary_image_np = np.uint8(image_np * 255)
        
        # Apply connected component labeling
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_image_np, connectivity=8)
        
        # Extract bounding boxes from stats
        bounding_boxes = []
        for stat in stats[1:]:  # Skip the first entry which is the background
            x, y, w, h, area = stat
            bounding_boxes.append((x, y, x + w, y + h))  # Format: (x_min, y_min, x_max, y_max)
        
        results.append((labels, bounding_boxes))
    
    return results

In [30]:
# Process in batches
batch_size = 100  # Adjust batch size as needed
num_batches = len(cvl_binarized_images) // batch_size + 1

cvl_results = []
for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, len(cvl_binarized_images))
    batch_images = cvl_binarized_images[start_idx:end_idx]
    
    batch_results = calculate_bounding_boxes(batch_images)
    cvl_results.extend(batch_results)

print("Bounding boxes calculated for CVL dataset.")


Bounding boxes calculated for CVL dataset.


In [31]:
# Process in batches
batch_size = 100  # Adjust batch size as needed
num_batches = len(iam_binarized_images) // batch_size + 1

iam_results = []
for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, len(iam_binarized_images))
    batch_images = iam_binarized_images[start_idx:end_idx]
    
    batch_results = calculate_bounding_boxes(batch_images)
    iam_results.extend(batch_results)

print("Bounding boxes calculated for IAM dataset.")


Bounding boxes calculated for IAM dataset.


Drawing Bounding Boxes

In [1]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Function to load and invert a single binary image
def load_binary_image(file_path):
    try:
        # Load the binary image tensor
        image_tensor = torch.load(file_path)
        if isinstance(image_tensor, torch.Tensor) and image_tensor.dim() == 3:
            # Convert tensor to numpy array
            image_array = image_tensor.numpy()
            # Ensure dtype is uint8 (0-255)
            image_array = np.uint8(image_array * 255)
            # Save the plot as an image file
            plt.savefig(save_path, bbox_inches='tight', pad_inches=0)
            plt.close()
            return 
        else:
            print(f"Unsupported tensor shape: {image_tensor.shape}")
            return None
    except Exception as e:
        print(f"Error loading binary image from {file_path}: {e}")
        return None

# Function to overlay bounding boxes on an image
def overlay_bounding_boxes(image, bounding_boxes, save_path, box_color='r'):
    list_of_tuple = bounding_boxes
    fig, ax = plt.subplots(1)
    ax.imshow(image, cmap='gray')
    
    for bbox in list_of_tuple:
        x_min, y_min, x_max, y_max = bbox
        rect = patches.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min, fill=False, edgecolor=box_color, linewidth=1)
        ax.add_patch(rect)
    
    ax.axis('off')
    
    # Save the plot as an image file
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0)
    plt.close()

def process_folder(binary_image_file_path, bounding_box, output_folder, prefix):
    # Load binary images
    binary_images = load_binary_image(binary_image_file_path)
    if binary_images is None:
        return

    # Ensure that the number of bounding boxes matches the number of images
    num_images = binary_images.shape[0]
    if len(bounding_box) != num_images:
        print(f"Warning: Number of bounding boxes ({len(bounding_box)}) does not match number of images ({num_images}).")
        return

    
    # Process each image and its corresponding bounding boxes
    for i in range(num_images):
        image = binary_images[i]
        bboxes = bounding_box[i][1]
        filename = f'{prefix}_{i}.png'
        save_path = os.path.join(output_folder, filename)
       
        # Overlay bounding boxes on the image
        overlay_bounding_boxes(image, bboxes, save_path)
        print(f"Processed {filename}")

In [32]:
import cv2
import os
import numpy as np
from PIL import Image, ImageOps

In [33]:
def resize_image(image, size=(128, 128)):
    return cv2.resize(image, size)

def resize_and_clean_words(crop_folder):
    if not os.path.exists(crop_folder):
        raise ValueError(f"The folder '{crop_folder}' does not exist.")

    for root, dirs, files in os.walk(crop_folder):
        for img_file in files:
            if img_file.endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(root, img_file)
                image = cv2.imread(img_path)
                
                # Resize the image
                resized_image = resize_image(image)
                
                # Save the resized image (overwrite the original cropped image)
                cv2.imwrite(img_path, resized_image)
                print(f"Resized image saved to {img_path}")

In [42]:
def load_image_tensor(file_path):
    try:
        # Load the tensor from the .pt file
        image_tensor = torch.load(file_path)
        if isinstance(image_tensor, torch.Tensor):
            # Convert tensor to numpy array
            image_array = image_tensor.numpy()
            
            # If the tensor is in (C, H, W) format, convert it to (H, W, C)
            if image_array.ndim == 4:  # (N, C, H, W) format
                image_array = np.transpose(image_array, (0, 2, 3, 1))
            
            return image_array
        else:
            print(f"Unsupported tensor type: {type(image_tensor)}")
            return None
    except Exception as e:
        print(f"Error loading image tensor from {file_path}: {e}")
        return None

def process_the_folder(binary_image_file_path, bounding_box, output_folder, prefix):
    # Load binary images
    binary_images = load_image_tensor(binary_image_file_path)
    if binary_images is None:
        return

    # Ensure that the number of bounding boxes matches the number of images
    num_images = binary_images.shape[0]
    if len(bounding_box) != num_images:
        print(f"Warning: Number of bounding boxes ({len(bounding_box)}) does not match number of images ({num_images}).")
        return

    
    # Process each image and its corresponding bounding boxes
    for i in range(num_images):
        image = binary_images[i]
        bboxes = bounding_box[i][1]
        img_subfolder = os.path.join(output_folder, f'{prefix}_{i}')
        
        # Create a subfolder for each image
        if not os.path.exists(img_subfolder):
            os.makedirs(img_subfolder)
        
        # Crop and save bounding boxes in the subfolder
        crop(image, bboxes, img_subfolder)
        

def crop(image, bounding_boxes, save_folder):
    # Create a blank white image with the same dimensions
    output_image = np.ones_like(image) * 255

    # Draw each bounding box on the output image
    for bbox in bounding_boxes:
        x, y, w, h = bbox
        x, y = max(x, 0), max(y, 0)
        w = min(w, image.shape[1])
        h = min(h, image.shape[0])
        # Crop the word area
        word_crop = image[y:y+h, x:x+w]
        output_image[y:y+h, x:x+w] = word_crop

    # Save the cropped image
    crop_path = os.path.join(save_folder, f"word_{i}.png")
    if not cv2.imwrite(crop_path, word_crop):
        print(f"Failed to save cropped image to {crop_path}")
    else:
        print(f"Cropped image saved to {crop_path}")

In [43]:
CVL_file_path = '/root/autodl-tmp/APS360_Project/Machine_Learning_Output/binary_image_bounding_box/CVL_binary.pt'
IAM_file_path = '/root/autodl-tmp/APS360_Project/Machine_Learning_Output/binary_image_bounding_box/IAM_binary.pt'

output_folder1 = '/root/autodl-tmp/APS360_Project/Machine_Learning_Output/output/CVL_rec'
output_folder2 = '/root/autodl-tmp/APS360_Project/Machine_Learning_Output/output/IAM_rec'
process_the_folder(CVL_file_path, cvl_results ,output_folder1, 'CVL')

Cropped image saved to /root/autodl-tmp/APS360_Project/Machine_Learning_Output/output/CVL_rec/CVL_0/word_15.png
Cropped image saved to /root/autodl-tmp/APS360_Project/Machine_Learning_Output/output/CVL_rec/CVL_1/word_15.png
Cropped image saved to /root/autodl-tmp/APS360_Project/Machine_Learning_Output/output/CVL_rec/CVL_2/word_15.png
Cropped image saved to /root/autodl-tmp/APS360_Project/Machine_Learning_Output/output/CVL_rec/CVL_3/word_15.png
Cropped image saved to /root/autodl-tmp/APS360_Project/Machine_Learning_Output/output/CVL_rec/CVL_4/word_15.png
Cropped image saved to /root/autodl-tmp/APS360_Project/Machine_Learning_Output/output/CVL_rec/CVL_5/word_15.png
Cropped image saved to /root/autodl-tmp/APS360_Project/Machine_Learning_Output/output/CVL_rec/CVL_6/word_15.png
Cropped image saved to /root/autodl-tmp/APS360_Project/Machine_Learning_Output/output/CVL_rec/CVL_7/word_15.png
Cropped image saved to /root/autodl-tmp/APS360_Project/Machine_Learning_Output/output/CVL_rec/CVL_8/word

In [ ]:
process_the_folder(IAM_file_path, iam_results, output_folder2, 'IAM')

In [12]:
def crop_black_background(image_path):
    """Crop out the black background from an image and return the processed image."""
    try:
        # Open the image
        image = Image.open(image_path)
        
        # Check if image has an alpha channel
        if image.mode == 'RGBA':
            r, g, b, a = image.split()
            rgb_image = Image.merge('RGB', (r, g, b))
        else:
            rgb_image = image.convert('RGB')

        # Invert the image
        inverted_image = ImageOps.invert(rgb_image)
        
        # Re-add the alpha channel if it was originally present
        if image.mode == 'RGBA':
            r2, g2, b2 = inverted_image.split()
            final_image = Image.merge('RGBA', (r2, g2, b2, a))
        else:
            final_image = inverted_image.convert('RGBA')
        
        return final_image

    except Exception as e:
        print(f"Error processing image {image_path}: {e}")
        return None

def process_images_in_folder(folder_path):
    """Process all images in a folder, crop out the black background, and overwrite the original images."""
    # List all files in the folder
    for file_name in os.listdir(folder_path):
        if file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            # Construct full file path
            img_path = os.path.join(folder_path, file_name)
            
            # Process image
            cropped_image = crop_black_background(img_path)
            
            if cropped_image:
                # Save the processed image by overwriting the original image
                cropped_image.save(img_path)
                print(f"Processed and saved {img_path}")

In [ ]:
folder_path = '/root/autodl-tmp/APS360_Project/Machine_Learning_Output/output/CVL'  # Path to the folder containing images

# Process all images in the folder
process_images_in_folder(folder_path)

In [ ]:
folder_path = '/root/autodl-tmp/APS360_Project/Machine_Learning_Output/output/IAM'  # Path to the folder containing images

# Process all images in the folder
process_images_in_folder(folder_path)